# Moce zainstalowane: baza bloków vs ENTSO-E

Porównuję moce zainstalowane z pliku `data/power_units_PL_DE_CZ_SK_LT_SE.csv` z mocami zainstalowanymi z ENTSO-E Transparency Platform
(Installed Capacity per Production Type, rok 2025) dla PL, DE, CZ, SK, LT i SE.

Kroki:
1. Wczytanie bazy bloków i zsumowanie mocy po kraju i grupie paliw
2. Pobranie danych z ENTSO-E (przez bibliotekę `entsoe-py`)
3. Porównanie: tabela kraj × paliwo, różnica w MW i %
4. Wykresy słupkowe per kraj

Klucz API jest w pliku `.env` w głównym folderze projektu (linia `ENTSOE_API_KEY=...`), nigdy w kodzie.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import os
from entsoe import EntsoePandasClient

# Wczytanie danych

Wczytuję bazę bloków (492 jednostki). Ścieżka zaczyna się od `../`, bo notatnik leży w folderze `notebooks/`.

In [2]:
power_units = pd.read_csv("../data/power_units_PL_DE_CZ_SK_LT_SE.csv")

In [3]:
power_units.head()

,unit_name,unit_id,country_code,installed_capacity_MW,efficiency,fuel_type,category,is_chp,source_capacity_status,source_efficiency,source_is_chp,source_unit_id,efficiency_1,efficiency_1_source
0,Ledvice B6,OEU-453,CZ,660.0,0.421,lignite,lignite,0,GEM Wiki - Ledvice power station (edited 25.10...,0.421 = 0.357 / 0.847 t CO2/MWh - unit emissio...,own assessment - condensing system power plant,unit id in OPSD EU (OEU-453),0.425,"""up to 42.5%"" (net/gross not stated) - oEnerge..."
1,Tisova Phase,PPM-1258,CZ,276.0,0.330,lignite,lignite,0,"powerplantmatching v0.9.1 (PyPSA, data as of A...","class ""old lignite unit"" - Instrat Foundation,...",own assessment - condensing system power plant,powerplantmatching id (plant-level record),NaN,NaN
2,Prunerov 2 B23,OEU-500,CZ,250.0,0.389,lignite,lignite,0,GEM Wiki - Prunerov power station (edited 17.1...,0.389 = 0.357 / 0.918 t CO2/MWh - unit emissio...,own assessment - condensing system power plant,unit id in OPSD EU (OEU-500),NaN,NaN
3,Prunerov 2 B24,OEU-501,CZ,250.0,0.389,lignite,lignite,0,GEM Wiki - Prunerov power station (edited 17.1...,0.389 = 0.357 / 0.918 t CO2/MWh - unit emissio...,own assessment - condensing system power plant,unit id in OPSD EU (OEU-501),NaN,NaN
4,Prunerov 2 B25,OEU-502,CZ,250.0,0.389,lignite,lignite,0,GEM Wiki - Prunerov power station (edited 17.1...,0.389 = 0.357 / 0.918 t CO2/MWh - unit emissio...,own assessment - condensing system power plant,unit id in OPSD EU (OEU-502),NaN,NaN


In [4]:
power_units.info()

<class 'pandas.DataFrame'>
RangeIndex: 492 entries, 0 to 491
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   unit_name               492 non-null    str    
 1   unit_id                 492 non-null    str    
 2   country_code            492 non-null    str    
 3   installed_capacity_MW   492 non-null    float64
 4   efficiency              492 non-null    float64
 5   fuel_type               492 non-null    str    
 6   category                492 non-null    str    
 7   is_chp                  492 non-null    int64  
 8   source_capacity_status  492 non-null    str    
 9   source_efficiency       492 non-null    str    
 10  source_is_chp           492 non-null    str    
 11  source_unit_id          492 non-null    str    
 12  efficiency_1            125 non-null    float64
 13  efficiency_1_source     125 non-null    str    
dtypes: float64(3), int64(1), str(10)
memory usage: 53.9 K

Sprawdzam, jakie wartości są w kolumnach `fuel_type` i `category`.
`fuel_type` ma czasem kilka typów naraz (np. `gas - CCGT / gas - steam turbine`), a `category` ma jedną krótką nazwę, więc do mapowania biorę `category`.

In [5]:
power_units["fuel_type"].value_counts()

fuel_type
hard coal                                                                                                                       112
gas - CCGT                                                                                                                      100
lignite                                                                                                                          84
gas - steam turbine (gas-fired boiler)                                                                                           57
oil                                                                                                                              44
solid biomass                                                                                                                    38
nuclear (uranium)                                                                                                                18
gas - OCGT                                                        

In [6]:
power_units["category"].value_counts()

category
hard_coal         109
gas_ccgt           97
lignite            82
gas_other          63
oil                43
biomass            36
nuclear            18
gas_ocgt            8
hydro               6
pumped_storage      6
battery             6
chp_gas             5
chp_biomass         4
chp_hard_coal       3
gas                 3
chp_lignite         2
chp_oil             1
Name: count, dtype: int64

In [7]:
power_units["country_code"].value_counts()

country_code
DE    231
PL    115
CZ     67
SE     51
SK     18
LT     10
Name: count, dtype: int64

Bloki DE w rezerwie sieciowej mają w nazwie `[grid reserve]`. Zostawiam je w sumie (ENTSO-E prawdopodobnie też liczy je w mocy zainstalowanej – do sprawdzenia),
ale pokazuję, ile to MW, żeby było wiadomo, skąd może brać się różnica.

In [8]:
rezerwa_sieciowa = power_units[power_units["unit_name"].str.contains("[grid reserve]", regex=False)]
print("Liczba bloków w rezerwie sieciowej:", len(rezerwa_sieciowa))
rezerwa_sieciowa.groupby(["country_code", "category"])["installed_capacity_MW"].sum()

Liczba bloków w rezerwie sieciowej: 28


country_code  category 
DE            gas_ccgt      726.0
              gas_ocgt      147.0
              gas_other     644.0
              hard_coal    7846.0
              oil           857.0
Name: installed_capacity_MW, dtype: float64

## Mapa kategorii na wspólne grupy

Obie strony (baza bloków i ENTSO-E) przypisuję do tych samych grup o polskich nazwach. Najpierw słownik dla kolumny `category`.
Kategorie `chp_...` (elektrociepłownie) idą do grupy swojego paliwa.

In [9]:
category_na_grupe = {
    "hard_coal": "węgiel kamienny",
    "chp_hard_coal": "węgiel kamienny",
    "lignite": "węgiel brunatny",
    "chp_lignite": "węgiel brunatny",
    "gas_ccgt": "gaz",
    "gas_ocgt": "gaz",
    "gas_other": "gaz",
    "gas": "gaz",
    "chp_gas": "gaz",
    "oil": "olej",
    "chp_oil": "olej",
    "nuclear": "jądrowe",
    "biomass": "biomasa",
    "chp_biomass": "biomasa",
    "hydro": "woda (bez ESP)",
    "pumped_storage": "ESP",
    "battery": "bateria",
}

In [10]:
power_units["grupa"] = power_units["category"].map(category_na_grupe)

# sprawdzenie: czy jakaś kategoria nie dostała grupy (powinno być 0 wierszy)
power_units[power_units["grupa"].isna()]

,unit_name,unit_id,country_code,installed_capacity_MW,efficiency,fuel_type,category,is_chp,source_capacity_status,source_efficiency,source_is_chp,source_unit_id,efficiency_1,efficiency_1_source,grupa


Sumuję moc zainstalowaną po kraju i grupie.

In [11]:
moce_power_units = power_units.groupby(["country_code", "grupa"])["installed_capacity_MW"].sum().reset_index()
moce_power_units = moce_power_units.rename(columns={"country_code": "kraj", "installed_capacity_MW": "MW_power_units"})
moce_power_units

,kraj,grupa,MW_power_units
0,CZ,ESP,1175.000
1,CZ,bateria,42.000
2,CZ,biomasa,30.000
3,CZ,gaz,1818.600
4,CZ,jądrowe,4164.000
5,CZ,woda (bez ESP),674.600
6,CZ,węgiel brunatny,6737.500
7,CZ,węgiel kamienny,667.200
8,DE,ESP,7230.500
9,DE,bateria,21951.000


In [12]:
# suma per kraj, dla kontroli
moce_power_units.groupby("kraj")["MW_power_units"].sum()

kraj
CZ     15308.900
DE    112863.328
LT      2635.000
PL     34500.200
SE     30014.100
SK      6825.200
Name: MW_power_units, dtype: float64

# Pobranie danych z ENTSO-E

Klucz API wczytuję z pliku `.env` (w głównym folderze projektu, czyli `../.env`). Funkcja `load_dotenv` wczytuje plik,
a `os.getenv` zwraca wartość zmiennej. Klucza nie wypisuję, sprawdzam tylko, czy się wczytał.

In [ ]:
load_dotenv("../.env")
klucz_api = os.getenv("ENTSOE_API_KEY")
print("Klucz wczytany:", klucz_api is not None)

In [ ]:
client = EntsoePandasClient(api_key=klucz_api)

Kody krajów w `entsoe-py`. Dla DE używam kodu kraju `"DE"` (Niemcy). Jeśli zapytanie zwróci błąd lub puste dane, można spróbować strefy `"DE_LU"`.
SE pobieram dla całego kraju (`"SE"`), bo baza bloków ma kod kraju, a nie strefy SE1–SE4.

Moc zainstalowana w ENTSO-E jest podawana raz na rok, więc biorę zakres od 1.01.2025 do 31.12.2025.

In [ ]:
kraje_entsoe = {
    "PL": "PL",
    "DE": "DE",
    "CZ": "CZ",
    "SK": "SK",
    "LT": "LT",
    "SE": "SE",
}

poczatek = pd.Timestamp("2025-01-01", tz="Europe/Brussels")
koniec = pd.Timestamp("2025-12-31", tz="Europe/Brussels")

Pobieram moc zainstalowaną per typ produkcji dla każdego kraju i zapisuję w słowniku `{kraj: df}`.
Każde zapytanie trwa kilka sekund.

In [ ]:
moce_entsoe = {}
for kraj, kod_entsoe in kraje_entsoe.items():
    print("Pobieram:", kraj)
    moce_entsoe[kraj] = client.query_installed_generation_capacity(kod_entsoe, start=poczatek, end=koniec)

Sprawdzenie: jak wygląda wynik dla jednego kraju. Wiersz = rok, kolumny = typy produkcji ENTSO-E, wartości w MW.

In [ ]:
moce_entsoe["PL"]

In [ ]:
# sprawdzenie: ile wierszy (lat) przyszło dla każdego kraju (powinien być 1 wiersz = 2025)
for kraj, moce in moce_entsoe.items():
    print(kraj, moce.shape, list(moce.index))

Zamieniam słownik na jedną długą tabelę: kolumny `kraj`, `typ_entsoe`, `MW_entsoe`. Biorę pierwszy wiersz (`.iloc[0]`), czyli rok 2025.

In [ ]:
wiersze = []
for kraj, moce in moce_entsoe.items():
    for typ_entsoe in moce.columns:
        wiersze.append({"kraj": kraj, "typ_entsoe": typ_entsoe, "MW_entsoe": moce[typ_entsoe].iloc[0]})

moce_entsoe_tabela = pd.DataFrame(wiersze)
moce_entsoe_tabela

Zapisuję pobrane dane do CSV, żeby następnym razem nie pytać API (folder `data/` nie trafia do gita).

In [ ]:
moce_entsoe_tabela.to_csv("../data/moce_zainstalowane_entsoe_2025.csv", index=False)

## Mapa typów produkcji ENTSO-E na wspólne grupy

- `Fossil Coal-derived gas` (CZ Vřesová) liczę jako gaz.
- Wiatr, PV, odpady, geotermia, `Other` i `Other renewable` pomijam, bo nie ma ich w bazie bloków.
- Uwaga: dla SE kategoria `Other` to biomasa + olej + gaz, więc w SE te grupy w ENTSO-E mogą wyjść za małe.

In [ ]:
entsoe_na_grupe = {
    "Fossil Hard coal": "węgiel kamienny",
    "Fossil Brown coal/Lignite": "węgiel brunatny",
    "Fossil Gas": "gaz",
    "Fossil Coal-derived gas": "gaz",
    "Fossil Oil": "olej",
    "Nuclear": "jądrowe",
    "Biomass": "biomasa",
    "Hydro Run-of-river and pondage": "woda (bez ESP)",
    "Hydro Water Reservoir": "woda (bez ESP)",
    "Hydro Pumped Storage": "ESP",
    "Energy storage": "bateria",
}

In [ ]:
moce_entsoe_tabela["grupa"] = moce_entsoe_tabela["typ_entsoe"].map(entsoe_na_grupe)

# sprawdzenie: które typy pomijam (brak grupy)
pominiete = moce_entsoe_tabela[moce_entsoe_tabela["grupa"].isna()]
pominiete

In [ ]:
moce_entsoe_grupy = moce_entsoe_tabela.groupby(["kraj", "grupa"])["MW_entsoe"].sum().reset_index()
moce_entsoe_grupy

# Porównanie

Łączę obie tabele po kraju i grupie (`how="outer"`, żeby zostały też grupy, które są tylko po jednej stronie). Brak wartości = 0 MW.

In [ ]:
porownanie = pd.merge(moce_power_units, moce_entsoe_grupy, on=["kraj", "grupa"], how="outer")
porownanie["MW_power_units"] = porownanie["MW_power_units"].fillna(0)
porownanie["MW_entsoe"] = porownanie["MW_entsoe"].fillna(0)
porownanie

Różnica = baza bloków − ENTSO-E. Dodatnia różnica znaczy, że w bazie bloków jest więcej MW niż w ENTSO-E.
Procent liczę względem ENTSO-E, tylko tam, gdzie ENTSO-E ma więcej niż 0 MW (żeby nie dzielić przez zero).

In [ ]:
porownanie["roznica_MW"] = porownanie["MW_power_units"] - porownanie["MW_entsoe"]

porownanie["roznica_%"] = np.nan
czy_entsoe_powyzej_zera = porownanie["MW_entsoe"] > 0
porownanie.loc[czy_entsoe_powyzej_zera, "roznica_%"] = porownanie["roznica_MW"] / porownanie["MW_entsoe"] * 100

porownanie = porownanie.round(1)
porownanie

Duże rozbieżności (więcej niż ±20% albo ±500 MW):

In [ ]:
porownanie[(porownanie["roznica_%"].abs() > 20) | (porownanie["roznica_MW"].abs() > 500)]

Tabela kraj × grupa z różnicą w MW (`pivot`).

In [ ]:
porownanie.pivot(index="grupa", columns="kraj", values="roznica_MW")

Sumy per kraj (tylko grupy z mapy, czyli bez wiatru i PV).

In [ ]:
sumy_kraje = porownanie.groupby("kraj")[["MW_power_units", "MW_entsoe", "roznica_MW"]].sum()
sumy_kraje

In [ ]:
porownanie.to_csv("../data/porownanie_mocy_power_units_entsoe_2025.csv", index=False)

# Wykresy

Dla każdego kraju wykres słupkowy: dwa słupki obok siebie dla każdej grupy paliw (baza bloków i ENTSO-E).
`np.arange` daje pozycje 0, 1, 2, …, a słupki przesuwam o pół szerokości w lewo i w prawo.

In [ ]:
szerokosc = 0.4

for kraj in kraje_entsoe:
    dane_kraju = porownanie[porownanie["kraj"] == kraj]
    pozycje = np.arange(len(dane_kraju))

    plt.figure(figsize=(18, 6))
    plt.bar(pozycje - szerokosc / 2, dane_kraju["MW_power_units"], width=szerokosc, label="baza bloków (power_units)")
    plt.bar(pozycje + szerokosc / 2, dane_kraju["MW_entsoe"], width=szerokosc, label="ENTSO-E 2025")
    plt.xticks(pozycje, dane_kraju["grupa"])
    plt.title("Moc zainstalowana – " + kraj)
    plt.ylabel("MW")
    plt.legend()
    plt.show()

Różnica w % dla wszystkich krajów na jednym wykresie (słupki dla każdej grupy, kolory = kraje).

In [ ]:
grupy = list(category_na_grupe.values())
grupy_bez_powtorzen = []
for grupa in grupy:
    if grupa not in grupy_bez_powtorzen:
        grupy_bez_powtorzen.append(grupa)

pozycje = np.arange(len(grupy_bez_powtorzen))
szerokosc_kraju = 0.13

plt.figure(figsize=(18, 6))
numer_kraju = 0
for kraj in kraje_entsoe:
    dane_kraju = porownanie[porownanie["kraj"] == kraj]
    dane_kraju = dane_kraju.set_index("grupa").reindex(grupy_bez_powtorzen)
    plt.bar(pozycje + numer_kraju * szerokosc_kraju, dane_kraju["roznica_%"], width=szerokosc_kraju, label=kraj)
    numer_kraju = numer_kraju + 1

plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(pozycje + 2.5 * szerokosc_kraju, grupy_bez_powtorzen)
plt.title("Różnica mocy: baza bloków vs ENTSO-E 2025")
plt.ylabel("różnica [%]")
plt.legend()
plt.show()

# Porównanie z ENTSO-E per jednostka (Installed Capacity per Production Unit)

ENTSO-E publikuje też moc zainstalowaną per **jednostka wytwórcza** (production unit, TR 14.1.B).
Uwaga: jednostka ENTSO-E to często **cała elektrownia albo jej część, a nie jeden blok**, np. „Turów” 1830 MW (u mnie 7 bloków Turow B01–B07),
„Jaworzno 3” 1257,8 MW (u mnie Jaworzno 3 B1–B6). W ENTSO-E są też jednostki poniżej 100 MW (np. Karolin 1: 44,8 MW).

Funkcja `client.query_installed_generation_capacity_per_unit` z `entsoe-py` wywala się (`'NoneType' object has no attribute 'text'`), bo przy niektórych jednostkach brakuje w danych jakiegoś pola.
Dlatego pobieram surowy XML przez `EntsoeRawClient` i sam wyciągam z niego tylko to, czego potrzebuję:
kod EIC, nazwę, moc, strefę i typ produkcji. Typ jest potrzebny, żeby odfiltrować wiatr i PV, których nie ma w bazie bloków.

Słownik: kod EIC strefy → nazwa strefy (takie nazwy jak w moich danych).

In [ ]:
kod_strefy_na_nazwe = {
    "10YPL-AREA-----S": "PL",
    "10Y1001A1001A82H": "DE-LU",
    "10YCZ-CEPS-----N": "CZ",
    "10YSK-SEPS-----K": "SK",
    "10YLT-1001A0008Q": "LT",
    "10Y1001A1001A44P": "SE1",
    "10Y1001A1001A45N": "SE2",
    "10Y1001A1001A46L": "SE3",
    "10Y1001A1001A47J": "SE4",
}

Pobieram surowy XML dla każdego kraju i zapisuję w słowniku `{kraj: tekst}`.

Dla DE moce per blok są w ENTSO-E publikowane dla strefy `DE_LU`, a nie dla kraju `"DE"` (zapytanie o `"DE"` daje `NoMatchingDataError`, czyli „brak danych”).
Są też dostępne dla czterech operatorów (50Hertz, Amprion, TenneT, TransnetBW), ale to te same bloki, więc biorę tylko `DE_LU`, żeby nie liczyć ich dwa razy.
Uwaga: `DE_LU` może zawierać też bloki z Luksemburga (np. ESP Vianden), których nie ma w bazie bloków DE – sprawdzić w grupie ESP.

`try:` / `except NoMatchingDataError:` znaczy: spróbuj pobrać, a jeśli dla danego kraju nie ma danych, wypisz to i idź dalej, zamiast zatrzymywać notatnik.

In [ ]:
from entsoe import EntsoeRawClient
from entsoe.exceptions import NoMatchingDataError

raw_client = EntsoeRawClient(api_key=klucz_api)

# kody krajów dla danych per blok: DE jest publikowane dla strefy DE_LU, a nie dla kraju "DE"
kraje_entsoe_bloki = {
    "PL": "PL",
    "DE": "DE_LU",
    "CZ": "CZ",
    "SK": "SK",
    "LT": "LT",
    "SE": "SE",
}

xml_jednostki = {}
for kraj, kod_entsoe in kraje_entsoe_bloki.items():
    print("Pobieram:", kraj)
    try:
        xml_jednostki[kraj] = raw_client.query_installed_generation_capacity_per_unit(kod_entsoe, start=poczatek, end=koniec)
    except NoMatchingDataError:
        # ENTSO-E odpowiedziało, ale nie ma danych dla tego kodu - pomijam kraj i idę dalej
        print("  brak danych dla", kod_entsoe)

In [ ]:
# sprawdzenie: długość tekstu dla każdego kraju (0 albo bardzo mało = brak danych)
for kraj, xml in xml_jednostki.items():
    print(kraj, len(xml))

Wyciągam z XML jeden wiersz na blok. `BeautifulSoup` zamienia tekst XML na obiekt, w którym szukam pól przez `.find("nazwa_pola")`.
Każdy blok to jeden `TimeSeries`. Jeśli brakuje pola ze strefą albo nazwą, wpisuję `"brak"`, zamiast wywalać błąd.
`PSRTYPE_MAPPINGS` zamienia kod typu (`B04`) na nazwę (`Fossil Gas`).

Nazwa bloku jest potrzebna do porównania blok do bloku (na końcu notatnika). Funkcja `popraw_kodowanie` naprawia polskie i czeskie litery:
tekst z API bywa odczytany w złym kodowaniu (zamiast `ł` wychodzi `Å‚`). Jeśli tekst jest już dobry, funkcja zwraca go bez zmian.

In [ ]:
from bs4 import BeautifulSoup
from entsoe.mappings import PSRTYPE_MAPPINGS


def popraw_kodowanie(tekst):
    # tekst odczytany jako latin-1 zamieniam z powrotem na utf-8; jeśli się nie da, zostawiam bez zmian
    try:
        return tekst.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return tekst


wiersze = []
for kraj, xml in xml_jednostki.items():
    dokument = BeautifulSoup(xml, "html.parser")
    for seria in dokument.find_all("timeseries"):
        # strefa: jeśli w danych jej nie ma, wpisuję "brak"
        pole_strefy = seria.find("inbiddingzone_domain.mrid")
        if pole_strefy is None:
            strefa = "brak"
        else:
            strefa = kod_strefy_na_nazwe.get(pole_strefy.text, pole_strefy.text)

        # nazwa bloku: jeśli jej nie ma, wpisuję "brak"
        pole_nazwy = seria.find("registeredresource.name")
        if pole_nazwy is None:
            nazwa = "brak"
        else:
            nazwa = popraw_kodowanie(pole_nazwy.text)

        wiersze.append({
            "kraj": kraj,
            "eic": seria.find("registeredresource.mrid").text,
            "nazwa_entsoe": nazwa,
            "MW_entsoe": float(seria.find("quantity").text),
            "strefa": strefa,
            "typ_entsoe": PSRTYPE_MAPPINGS.get(seria.find("psrtype").text),
        })

jednostki_entsoe = pd.DataFrame(wiersze)
jednostki_entsoe

In [ ]:
# sprawdzenie: liczba bloków, strefy, suma MW per kraj
print(jednostki_entsoe.shape)
print(jednostki_entsoe["strefa"].value_counts())
jednostki_entsoe.groupby("kraj")["MW_entsoe"].sum()

In [ ]:
# sprawdzenie: czy jakiś blok jest dwa razy (ten sam kod EIC)
jednostki_entsoe[jednostki_entsoe["eic"].duplicated(keep=False)]

In [ ]:
jednostki_entsoe.to_csv("../data/moce_zainstalowane_entsoe_bloki_2025.csv", index=False)

Moc per strefa i typ (tu widać podział SE na SE1–SE4).

In [ ]:
jednostki_entsoe.pivot_table(index="typ_entsoe", columns="strefa", values="MW_entsoe", aggfunc="sum")

Przypisuję bloki ENTSO-E do tych samych grup co wcześniej (słownik `entsoe_na_grupe`). Typy bez grupy (wiatr, PV, odpady, inne) pomijam.

In [ ]:
jednostki_entsoe["grupa"] = jednostki_entsoe["typ_entsoe"].map(entsoe_na_grupe)

# sprawdzenie: które typy pomijam i ile to MW
jednostki_entsoe[jednostki_entsoe["grupa"].isna()].groupby(["kraj", "typ_entsoe"])["MW_entsoe"].sum()

In [ ]:
bloki_entsoe = jednostki_entsoe[jednostki_entsoe["grupa"].notna()]
bloki_entsoe.shape

Z bazy bloków pomijam rekordy zbiorcze (w nazwie `grouped`, np. „DE - battery (all battery storage, grouped)”),
bo to suma wielu małych instalacji, a nie jeden blok. Nie filtruję po mocy, bo w ENTSO-E są też jednostki poniżej 100 MW.

In [ ]:
czy_zbiorczy = power_units["unit_name"].str.contains("grouped", regex=False)
power_units[czy_zbiorczy][["unit_name", "country_code", "installed_capacity_MW"]]

In [ ]:
bloki_power_units = power_units[czy_zbiorczy == False]
print(bloki_power_units.shape)
bloki_power_units.groupby("country_code")["installed_capacity_MW"].sum()

## Suma per kraj i grupa

Najpierw ogólnie: suma MW per kraj i grupa paliw – bloki z bazy (bez rekordów zbiorczych) vs jednostki ENTSO-E.
Bloki z bazy, które są małe i nie mają odpowiednika w ENTSO-E, zawyżają tu stronę bazy – dokładniej widać to w porównaniu jednostka po jednostce niżej.

In [ ]:
mw_power_units = bloki_power_units.groupby(["country_code", "grupa"])["installed_capacity_MW"].sum().reset_index()
mw_power_units = mw_power_units.rename(columns={"country_code": "kraj", "installed_capacity_MW": "MW_power_units"})

mw_entsoe = bloki_entsoe.groupby(["kraj", "grupa"])["MW_entsoe"].sum().reset_index()

porownanie_jednostki = pd.merge(mw_power_units, mw_entsoe, on=["kraj", "grupa"], how="outer")
porownanie_jednostki = porownanie_jednostki.fillna(0)

porownanie_jednostki["roznica_MW"] = porownanie_jednostki["MW_power_units"] - porownanie_jednostki["MW_entsoe"]
porownanie_jednostki["roznica_%"] = np.nan
czy_entsoe_powyzej_zera = porownanie_jednostki["MW_entsoe"] > 0
porownanie_jednostki.loc[czy_entsoe_powyzej_zera, "roznica_%"] = porownanie_jednostki["roznica_MW"] / porownanie_jednostki["MW_entsoe"] * 100

porownanie_jednostki = porownanie_jednostki.round(1)
porownanie_jednostki

In [ ]:
porownanie_jednostki.pivot(index="grupa", columns="kraj", values="roznica_MW")

# Porównanie jednostka po jednostce

Cel: dla każdej jednostki ENTSO-E znaleźć odpowiadające jej bloki z bazy i porównać moc, np. „Turów: w ENTSO-E 1830 MW, u mnie 7 bloków = 1948 MW”.

Nie ma wspólnego ID (w bazie `OEU-...`/`PPM-...`, w ENTSO-E kod EIC), więc dopasowuję **po nazwie**:
1. upraszczam nazwy (małe litery, bez ogonków: `Turów` → `turow`),
2. dla każdego bloku z bazy szukam w ENTSO-E jednostki o najbardziej podobnej nazwie, tylko w tym samym kraju i tej samej grupie paliw,
3. **kilka bloków może trafić do jednej jednostki ENTSO-E** (np. Turow B01–B07 → Turów) – wtedy sumuję ich moc,
4. automatyczne dopasowanie może się mylić, więc wynik trzeba przejrzeć, a złe dopasowania poprawić w słowniku `poprawki_dopasowania`.

In [ ]:
import unicodedata
from difflib import SequenceMatcher

Funkcja upraszczająca nazwę. `unicodedata.normalize("NFKD", ...)` rozbija literę z ogonkiem na literę i ogonek (`ó` → `o` + `´`),
a `unicodedata.combining(znak)` mówi, czy znak to sam ogonek – takie znaki pomijam. Litery `ł` i `ß` nie rozkładają się w ten sposób, więc zamieniam je ręcznie.

In [ ]:
def uprosc_nazwe(nazwa):
    nazwa = nazwa.lower()
    nazwa = nazwa.replace("ł", "l")
    nazwa = nazwa.replace("ß", "ss")
    nazwa = unicodedata.normalize("NFKD", nazwa)

    wynik = ""
    for znak in nazwa:
        if not unicodedata.combining(znak):
            wynik = wynik + znak
    return wynik

In [ ]:
# sprawdzenie
print(uprosc_nazwe("Bełchatów B03"))
print(uprosc_nazwe("Łaziska 3"))
print(uprosc_nazwe("Počerady B2"))

Funkcja podobieństwa nazwy bloku z bazy i nazwy jednostki ENTSO-E (0 = zupełnie różne, 1 = identyczne). Liczę dwa podobieństwa i biorę większe:
- całych nazw: `Bełchatów B03` vs `Belchatow B03` → 1,0,
- **początku** nazwy bloku (tak długiego jak nazwa z ENTSO-E) z nazwą z ENTSO-E: `Turow B07` → `turow` vs `Turów` → `turow` → 1,0.
  Dzięki temu blok dopasuje się do całej elektrowni, gdy ENTSO-E podaje ją jako jedną jednostkę.

`SequenceMatcher(None, a, b).ratio()` z biblioteki standardowej `difflib` liczy, jak bardzo dwa napisy są podobne.

In [ ]:
def podobienstwo_nazw(nazwa_bloku, nazwa_entsoe):
    blok = uprosc_nazwe(nazwa_bloku)
    entsoe = uprosc_nazwe(nazwa_entsoe)

    podobienstwo_calych = SequenceMatcher(None, blok, entsoe).ratio()

    poczatek_bloku = blok[:len(entsoe)]
    podobienstwo_poczatku = SequenceMatcher(None, poczatek_bloku, entsoe).ratio()

    return max(podobienstwo_calych, podobienstwo_poczatku)

In [ ]:
# sprawdzenie
print(podobienstwo_nazw("Belchatow B03", "Bełchatów B03"))   # ten sam blok
print(podobienstwo_nazw("Belchatow B03", "Bełchatów B08"))   # inny blok tej samej elektrowni
print(podobienstwo_nazw("Turow B07", "Turów"))               # blok -> cała elektrownia
print(podobienstwo_nazw("Jaworzno 3 B5", "Jaworzno 3"))      # blok -> elektrownia Jaworzno 3
print(podobienstwo_nazw("Jaworzno 3 B5", "Jaworzno 2"))      # inna elektrownia
print(podobienstwo_nazw("Belchatow B03", "Kozienice B10"))   # zupełnie co innego

Dopasowanie automatyczne. Dla każdego bloku z bazy przeglądam jednostki ENTSO-E z tego samego kraju i tej samej grupy
i zapamiętuję tę o najbardziej podobnej nazwie. To dwie zwykłe pętle `for`, więc może chwilę potrwać.

In [ ]:
wiersze = []
for indeks, blok in bloki_power_units.iterrows():
    # kandydaci: jednostki ENTSO-E z tego samego kraju i tej samej grupy paliw
    kandydaci = bloki_entsoe[(bloki_entsoe["kraj"] == blok["country_code"]) & (bloki_entsoe["grupa"] == blok["grupa"])]

    najlepsze_podobienstwo = 0
    najlepszy_eic = None
    for indeks_entsoe, kandydat in kandydaci.iterrows():
        podobienstwo = podobienstwo_nazw(blok["unit_name"], kandydat["nazwa_entsoe"])
        if podobienstwo > najlepsze_podobienstwo:
            najlepsze_podobienstwo = podobienstwo
            najlepszy_eic = kandydat["eic"]

    wiersze.append({
        "unit_id": blok["unit_id"],
        "eic": najlepszy_eic,
        "podobienstwo_nazwy": round(najlepsze_podobienstwo, 2),
    })

dopasowanie = pd.DataFrame(wiersze)
dopasowanie

Dopasowania o podobieństwie nazwy poniżej progu uznaję za brak dopasowania (blok najpewniej nie ma odpowiednika w ENTSO-E).
Próg 0,6 jest przyjęty przeze mnie na oko – można go zmienić.

In [ ]:
prog_podobienstwa = 0.6

dopasowanie.loc[dopasowanie["podobienstwo_nazwy"] < prog_podobienstwa, "eic"] = None

print("Dopasowane:", dopasowanie["eic"].notna().sum(), "z", len(dopasowanie))

## Ręczne poprawki

Po przejrzeniu wyników (niżej) wpisz tu poprawki: `"unit_id z bazy": "kod EIC z ENTSO-E"`.
Jeśli bloku na pewno nie ma w ENTSO-E, wpisz `None`. Poprawki nadpisują dopasowanie automatyczne.
Kod EIC znajdziesz w tabeli `jednostki_entsoe` (np. `jednostki_entsoe[jednostki_entsoe["nazwa_entsoe"].str.contains("Jaworzno")]`)
albo na stronie ENTSO-E w kolumnie „Code”.

In [ ]:
poprawki_dopasowania = {
    # "OEU-123": "19W000000000XXXX",
    # "PPM-456": None,
}

for unit_id, eic in poprawki_dopasowania.items():
    dopasowanie.loc[dopasowanie["unit_id"] == unit_id, "eic"] = eic
    dopasowanie.loc[dopasowanie["unit_id"] == unit_id, "podobienstwo_nazwy"] = 1.0

print("Liczba poprawek:", len(poprawki_dopasowania))

## Lista bloków z dopasowaniem

Jeden wiersz = jeden blok z bazy i jednostka ENTSO-E, do której trafił. Tu najłatwiej sprawdzić, czy dopasowanie ma sens.

In [ ]:
bloki_dopasowane = pd.merge(dopasowanie, bloki_power_units[["unit_id", "unit_name", "country_code", "grupa", "installed_capacity_MW"]], on="unit_id")
bloki_dopasowane = pd.merge(bloki_dopasowane, bloki_entsoe[["eic", "nazwa_entsoe", "MW_entsoe"]], on="eic", how="left")
bloki_dopasowane = bloki_dopasowane.rename(columns={"country_code": "kraj", "installed_capacity_MW": "MW_power_units"})
bloki_dopasowane = bloki_dopasowane[["kraj", "grupa", "unit_id", "unit_name", "MW_power_units", "eic", "nazwa_entsoe", "MW_entsoe", "podobienstwo_nazwy"]]
bloki_dopasowane

In [ ]:
# sprawdzenie: bloki z PL (zmień "PL" na inny kraj)
bloki_dopasowane[bloki_dopasowane["kraj"] == "PL"].sort_values("nazwa_entsoe")

In [ ]:
# bloki bez odpowiednika w ENTSO-E
bez_dopasowania = bloki_dopasowane[bloki_dopasowane["eic"].isna()]
print("Bez dopasowania:", len(bez_dopasowania))
bez_dopasowania.sort_values(["kraj", "MW_power_units"], ascending=[True, False])

## Tabela jednostka po jednostce

Sumuję moc bloków z bazy dla każdej jednostki ENTSO-E (`groupby("eic")`) i porównuję z mocą z ENTSO-E.
Nazwy bloków sklejam w jeden napis (`", ".join(lista)`), żeby było widać, co trafiło do danej jednostki.

In [ ]:
wiersze = []
for eic in bloki_dopasowane["eic"].dropna().unique():
    bloki_tej_jednostki = bloki_dopasowane[bloki_dopasowane["eic"] == eic]
    wiersze.append({
        "eic": eic,
        "bloki_z_bazy": ", ".join(bloki_tej_jednostki["unit_name"]),
        "liczba_blokow": len(bloki_tej_jednostki),
        "MW_power_units": bloki_tej_jednostki["MW_power_units"].sum(),
        "min_podobienstwo_nazwy": bloki_tej_jednostki["podobienstwo_nazwy"].min(),
    })

suma_blokow_na_jednostke = pd.DataFrame(wiersze)
suma_blokow_na_jednostke

In [ ]:
jednostka_po_jednostce = pd.merge(bloki_entsoe[["kraj", "grupa", "eic", "nazwa_entsoe", "MW_entsoe", "strefa"]], suma_blokow_na_jednostke, on="eic", how="left")
jednostka_po_jednostce["liczba_blokow"] = jednostka_po_jednostce["liczba_blokow"].fillna(0)
jednostka_po_jednostce["MW_power_units"] = jednostka_po_jednostce["MW_power_units"].fillna(0)

jednostka_po_jednostce["roznica_MW"] = jednostka_po_jednostce["MW_power_units"] - jednostka_po_jednostce["MW_entsoe"]
jednostka_po_jednostce["roznica_%"] = np.nan
czy_entsoe_powyzej_zera = jednostka_po_jednostce["MW_entsoe"] > 0
jednostka_po_jednostce.loc[czy_entsoe_powyzej_zera, "roznica_%"] = jednostka_po_jednostce["roznica_MW"] / jednostka_po_jednostce["MW_entsoe"] * 100
jednostka_po_jednostce = jednostka_po_jednostce.round(1)
jednostka_po_jednostce.head()

Ocena każdej jednostki ENTSO-E (zwykła pętla z `if`):
- `brak w bazie` – do jednostki ENTSO-E nie trafił żaden blok z bazy,
- `zgodne` – moce różnią się najwyżej o 5% (albo o 10 MW przy małych jednostkach),
- `różna moc` – bloki są dopasowane, ale suma mocy różni się bardziej (możliwe przyczyny: moc brutto vs netto, brakujący albo nadmiarowy blok, złe dopasowanie).

Progi (5%, 10 MW) są przyjęte przeze mnie na oko – można je zmienić.

In [ ]:
prog_procent = 5
prog_MW = 10

oceny = []
for indeks, jednostka in jednostka_po_jednostce.iterrows():
    if jednostka["liczba_blokow"] == 0:
        ocena = "brak w bazie"
    elif abs(jednostka["roznica_%"]) <= prog_procent or abs(jednostka["roznica_MW"]) <= prog_MW:
        ocena = "zgodne"
    else:
        ocena = "różna moc"
    oceny.append(ocena)

jednostka_po_jednostce["ocena"] = oceny
jednostka_po_jednostce["ocena"].value_counts()

In [ ]:
# liczba ocen per kraj
jednostka_po_jednostce.pivot_table(index="kraj", columns="ocena", values="eic", aggfunc="count")

Kolejność kolumn do przeglądania: ENTSO-E, potem bloki z bazy, potem różnica i ocena.

In [ ]:
jednostka_po_jednostce = jednostka_po_jednostce[["kraj", "grupa", "eic", "nazwa_entsoe", "strefa", "MW_entsoe",
                                                 "bloki_z_bazy", "liczba_blokow", "MW_power_units",
                                                 "roznica_MW", "roznica_%", "min_podobienstwo_nazwy", "ocena"]]
jednostka_po_jednostce

In [ ]:
# podgląd jednego kraju (zmień "PL" na inny kraj)
jednostka_po_jednostce[jednostka_po_jednostce["kraj"] == "PL"].sort_values("MW_entsoe", ascending=False)

In [ ]:
jednostka_po_jednostce[jednostka_po_jednostce["ocena"] == "różna moc"].sort_values("roznica_MW")

In [ ]:
jednostka_po_jednostce[jednostka_po_jednostce["ocena"] == "brak w bazie"].sort_values(["kraj", "MW_entsoe"], ascending=[True, False])

In [ ]:
jednostka_po_jednostce.to_csv("../data/porownanie_jednostka_po_jednostce_entsoe_2025.csv", index=False)
bloki_dopasowane.to_csv("../data/dopasowanie_blokow_do_jednostek_entsoe_2025.csv", index=False)

Wykres: moc jednostki w ENTSO-E vs suma bloków z bazy (tylko jednostki z dopasowaniem). Punkty na przekątnej = ta sama moc.

In [ ]:
dopasowane = jednostka_po_jednostce[jednostka_po_jednostce["liczba_blokow"] > 0]

plt.figure(figsize=(8, 8))
for kraj in kraje_entsoe:
    dane_kraju = dopasowane[dopasowane["kraj"] == kraj]
    plt.scatter(dane_kraju["MW_entsoe"], dane_kraju["MW_power_units"], label=kraj)

najwieksza_moc = dopasowane[["MW_entsoe", "MW_power_units"]].max().max()
plt.plot([0, najwieksza_moc], [0, najwieksza_moc], color="black", linewidth=0.8)
plt.title("Moc jednostki: ENTSO-E vs suma bloków z bazy")
plt.xlabel("MW ENTSO-E")
plt.ylabel("MW baza bloków")
plt.legend()
plt.show()